In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

from scipy.ndimage import uniform_filter

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="Buoyancy",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Calculation Functions
def GetInputVariables(inputDataDirectory, timeString):

    ################################# DENSITY POTENTIAL TEMPERATURE
    theta_rho = CallVariable(ModelData, DataManager, timeString, 'theta_v') #*yes, named theta_v, but really theta_rho after condensation

    inputDictionary = {'theta_rho': theta_rho}
    return inputDictionary

def VariableCalculation(inputDictionary):
    [theta_rho] = (inputDictionary[k] for k in ['theta_rho'])
    theta_rho_mean = np.mean(theta_rho, axis=(1, 2), keepdims=True)
    g=9.81  #m/s^2
    buoyancy = g * (theta_rho - theta_rho_mean) / theta_rho_mean  # shape: (Nz, Ny, Nx)
    
    outputDictionary={'buoyancy2': buoyancy} #called it theta_v, but found out later it is density potential temperature
    return outputDictionary

In [ ]:
####################################
#RUNNING

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in tqdm(loop_elements, total=len(loop_elements)):
    if np.mod(t,1)==0: print(f'Current time {t}')

    #getting timestring for loading input data
    timeString = ModelData.timeStrings[t]

    #loading input variables
    inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)

    #calculating variables
    outputDictionary = VariableCalculation(inputDictionary)
    
    #outputting
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary)

In [ ]:
######################################################
#TESTING

In [ ]:
# t=100
# array = CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName='buoyancy2')

In [ ]:
######################################################
#TESTING

In [ ]:
# #testing difference buoyancy definitions

# #Functions
# def VariableCalculation(t, relative=None):
#     theta_rho = CallVariable(ModelData, DataManager, ModelData.timeStrings[t], variableName="theta_v")
#     kms = int(1000 / np.round(ModelData.dx))
#     g = 9.81  # m/s/s
#     coast = 128 * kms

#     if relative == 'coast':
#         theta_left = theta_rho[:, :, :coast]
#         mean_left = np.nanmean(theta_left, axis=(1, 2), keepdims=True)

#         theta_right = theta_rho[:, :, coast:]
#         mean_right = np.nanmean(theta_right, axis=(1, 2), keepdims=True)

#         buoyancy = np.empty_like(theta_rho)
#         buoyancy[:, :, :coast] = g * (theta_left - mean_left) / mean_left
#         buoyancy[:, :, coast:] = g * (theta_right - mean_right) / mean_right

#     elif isinstance(relative, (int, float)):
#         r = int(relative * kms)  # radius in grid points
#         Nz, Ny, Nx = theta_rho.shape

#         #Build padded array
#         theta_padded = np.pad(theta_rho, ((0,0),(r,r),(0,0)), mode='edge')
#         theta_padded = np.pad(theta_padded, ((0,0),(0,0),(r,r)), mode='wrap')

#         # Vectorized
#         size = 2 * r + 1
#         local_mean = uniform_filter(theta_padded, size=(1, size, size), mode='nearest')
#         local_mean = local_mean[:, r:r+Ny, r:r+Nx]
#         buoyancy = g * (theta_rho - local_mean) / local_mean

#         # #column-by-column
#         # buoyancy = np.empty_like(theta_rho)
#         # for ix in range(Nx):
#         #     for iy in range(Ny):
#         #         # Offset indices due to padding
#         #         ix_p = ix + r
#         #         iy_p = iy + r
#         #         local = theta_padded[:, iy_p - r:iy_p + r + 1, ix_p - r:ix_p + r + 1]
#         #         local_mean = np.nanmean(local, axis=(1, 2), keepdims=True)
#         #         buoyancy[:, iy, ix] = g * (theta_rho[:, iy, ix] - local_mean[:, 0, 0]) / local_mean[:, 0, 0]
#     elif relative is None:
#         theta_rho_mean = np.nanmean(theta_rho, axis=(1, 2), keepdims=True)
#         buoyancy = g * (theta_rho - theta_rho_mean) / theta_rho_mean

#     return buoyancy

# def MakePlots(dictionary,ylevel=100,zlevel=None):
#     #Setting up GetSlice function
#     if zlevel is None:
#         def GetSlice(var): return var[:, ylevel]
#         x=ModelData.xh; y=ModelData.zh
#         y_km = ModelData.yh[ylevel]
#         y_max = ModelData.yh[-1]
#         sliceString = f"[y={y_km:.0f}/{y_max:.0f} km]"
#         ylabel='z (km)';xlabel='x (km)'
#     elif ylevel is None:
#         def GetSlice(var): return var[zlevel]
#         x=ModelData.xh; y=ModelData.yh
#         z_m = ModelData.zh[zlevel]*1e3
#         sliceString = f"[z={z_m:.1f} m]"
#         ylabel='y (km)';xlabel='x (km)'
        

#     fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True, gridspec_kw={'wspace':0.03}) 
#     levels=np.linspace(-0.05,0.05,21)
#     cmap='RdBu_r';cmap='bwr'
#     label=f'B (m/s/s)'
    
#     # Top-left
#     axis=axes[0,0]
#     data=dictionary['buoyancy']
#     cf1 = axis.contourf(x,y,GetSlice(data), cmap=cmap,levels=levels,extend='both'); fig.colorbar(cf1, ax=axis, label="")
#     axis.set_title(f'Buoyancy{sliceString}\ncompared to initial vertical profile')
    
#     # Top-right
#     axis=axes[0,1]
#     data=dictionary['buoyancy2']
#     cf2 = axis.contourf(x,y,GetSlice(data), cmap=cmap,levels=levels,extend='both'); fig.colorbar(cf2, ax=axis, label=label)
#     axis.set_title(f'Buoyancy{sliceString}\ncompared to each time vertical profile')
    
#     # Bottom-left
#     axis=axes[1,0]
#     data=dictionary['buoyancy3']
#     cf2 = axis.contourf(x,y,GetSlice(data), cmap=cmap,levels=levels,extend='both'); fig.colorbar(cf2, ax=axis, label="")
#     axis.set_title(f'Buoyancy{sliceString}\ncompared to each time vertical profile\n(coast-relative)')
    
#     # Bottom-right
#     axis=axes[1,1]
#     data=dictionary['buoyancy4']
#     cf2 = axis.contourf(x,y,GetSlice(data), cmap=cmap,levels=levels,extend='both'); fig.colorbar(cf2, ax=axis, label=label)
#     axis.set_title(f'Buoyancy{sliceString}\ncompared to each time vertical profile\n(10km_window-relative)')
    
#     #Labels and Ticks
#     for i, axis in enumerate(axes.ravel()):
#         row = i // 2; col = i % 2
    
#         axis.axvline(128, color='black', zorder=10)
#         axis.set_ylabel(ylabel if col == 0 else '')
#         axis.set_xlabel(xlabel if row == 1 else '')
#         if zlevel is None:
#             axis.set_ylim(0,10)

#     return fig

In [ ]:
# #testing difference buoyancy definitions

# #Data Loading and Calculating
# variableNames = ['buoyancy','buoyancy2']
# t=200
# dictionary = {variableName: CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName=variableName) for variableName in variableNames}
# dictionary['buoyancy3']=VariableCalculation(t,relative='coast')
# dictionary['buoyancy4']=VariableCalculation(t,relative=10)

In [ ]:
#testing difference buoyancy definitions
#Plotting
# fig = MakePlots(dictionary,ylevel=100,zlevel=None)
# fig = MakePlots(dictionary,ylevel=None,zlevel=5)